# ch05 Bonus 14：OLMo3 实现

> 对照官方 `ch05/13_olmo3`
> **参考真实模型**：Allen AI 的 OLMo3（2025，完全开源）

## 一句话

OLMo3 强调**完全开源**（数据+代码+权重），架构上用 **RoPE + SwiGLU + QK-Norm**，并优化了注意力缩放策略。

## 相对 Llama 的特点

1. **完全开源**：训练数据（Tulu/Dolma）也公开，可复现
2. **QK-Norm**：与 Qwen3/Gemma3 一致的 2024 改进
3. **注意力缩放**：对注意力分数用额外的可学习缩放因子
4. **无 bias、GQA**：现代标配

> OLMo 的价值在于「科研可复现」——是研究 LLM 训练机制的理想基线模型。

In [ ]:
import torch
import torch.nn as nn

# OLMo3 的注意力缩放因子（区别于标准 1/sqrt(head_dim)）
class OLMo3AttentionScale(nn.Module):
    def __init__(self, head_dim):
        super().__init__()
        # 标准用固定的 1/sqrt(head_dim)，OLMo 加可学习缩放
        self.scale = nn.Parameter(torch.tensor(head_dim ** -0.5))

    def forward(self, attn_scores):
        return attn_scores * self.scale

scale_layer = OLMo3AttentionScale(128)
scores = torch.randn(2, 8, 16, 16)
scaled = scale_layer(scores)
print(f"初始 scale: {scale_layer.scale.item():.4f} (= 1/√128)")
print(f"缩放后分数方差: {scaled.var():.4f}")
print("\n💡 OLMo 把固定的 1/√d 换成可学习，让模型自适应缩放强度。")
print("   OLMo 的主要价值是完全开源（数据+权重+代码），便于科研复现。")